# Notebook to Explore Agent Design

In [1]:
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)

ANTHROPIC_KEY = config['ANTHROPIC_KEY']

In [2]:
SYSTEM_PROMPT = """You are a Stylist workingt at H&M. You have a strong sense of fashion and your role is to assist customers in their shopping experience. 
You provide personalize recommendations as well as guidance to optimize their customer experience.
As of today, you can provide assistance only for custonmers based in Portugal.

When looking for stores information:
- always look for closest store to customer address
- if the customer prefers to not disclose its address, look for all stores in given city and when listing them, always put at top a flagship store if available."""

## Langchain

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver  

client = MultiServerMCPClient(
    {
        "my_server": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()
print(tools)

[StructuredTool(name='recommender_tool', description="Function to call the Stylist API.\nThe API retrieves the customer information from Mongo collection\nbefore submitting data to a Two Tower Tensorflow model.\n\nThis tool should be used when customer ask for new product recommendation.\n\n:param customer_id: id of the customer\n:type customer_id: str\n:param top_k: how many similar items to return\n:type top_k: int\n:param exclude_purchased: Exclude past customer's purchases from recommendations\n:type exclude_purchased: bool\n:param url: url of the recommender API\n:type url: str\n:return: Recommendation results\n:rtype: dict", args_schema={'properties': {'customer_id': {'type': 'string'}, 'top_k': {'default': 5, 'type': 'integer'}, 'exclude_purchased': {'default': True, 'type': 'boolean'}, 'url': {'default': 'http://stylistai-api:8000/invoke', 'type': 'string'}}, 'required': ['customer_id'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_

In [4]:
llm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=ANTHROPIC_KEY)
agent = create_agent(
    model=llm, 
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver()
    )

In [5]:
user_prompt = "Can you recommend me new items to add to my wardrobe?"

response = await agent.ainvoke({"messages": [{"role": "user", "content": user_prompt}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

I'd be happy to help you discover new items for your wardrobe! To provide you with personalized recommendations, I'll need your customer ID.

Could you please provide your customer ID so I can access your profile and suggest items that match your style preferences and shopping history?


In [6]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "Yes, my ID is: 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

Perfect! Based on your style profile, I have some great recommendations for you:

## My Top Pick for You:

**🌟 Shaping Brazilian Briefs** (Article #694344001)
- **Category:** Lingerie & Underwear
- **Style:** High-waisted microfibre and lace briefs
- **Features:** 
  - Beautiful combination of microfibre and lace details
  - High waist design for comfort and coverage
  - Light shaping effect on tummy and hips
  - Lined gusset with half-string back
  - Perfect blend of style and functionality

This piece offers both elegance and comfort, making it a great foundational piece for your wardrobe. The high-waist design is very on-trend and provides a flattering silhouette under any outfit!

Would you like to see more recommendations, or would you like information about where you can find this item in a store near you?


In [7]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "Can you show me two other items?"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

I apologize, but it seems the recommendation system is currently returning only one item for your profile. This could be due to your specific style preferences or purchasing history being very focused, or it might be a temporary limitation.

However, I'd love to help you explore more options! Could you tell me:
- What type of items are you most interested in? (e.g., tops, dresses, outerwear, accessories)
- Are there any specific occasions you're shopping for?
- Any particular styles or colors you're drawn to right now?

This will help me provide you with more tailored suggestions and assistance in finding the perfect pieces for your wardrobe!


In [8]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "I see. Where could I find this item? Is there a store near me?"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

I'd be happy to help you find a store near you where you can check out this item! 

To locate the closest H&M store, could you please provide me with either:
- Your full address (for the most accurate result), or
- The city where you'd like to shop

This way I can direct you to the nearest location!


In [9]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "I live in Braga"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

Great! I found **2 H&M stores in Braga** where you can find the Shaping Brazilian Briefs:

## 📍 H&M Stores in Braga:

### 1. **H&M C.C. Braga Parque**
- **Address:** R. dos Congregados 1, 4710-427 Braga
- **Phone:** +351-800780330
- **Opening Hours:** 10:00-23:00 (Monday to Sunday)

### 2. **H&M C.C. Nova Arcada**
- **Address:** Avenida de Lamas, nº 100, Centro Comercial Nova Arcada, 4700-084 Braga
- **Phone:** +351-800780330
- **Opening Hours:** 10:00-23:00 (Monday to Sunday)

Both stores are conveniently located in shopping centers with great opening hours throughout the week! 

**Pro tip:** I'd recommend calling ahead (+351-800780330) to check if they have the specific item (Article #694344001) in your size in stock before visiting.

Is there anything else I can help you with today?


In [10]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "Ok I am staying at the hotel, which store is the closest to me right now? I am at Largo Carlos Amarante 150, 4700-308 Braga, Portugal"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

Perfect! The closest H&M store to your hotel is:

## 📍 **H&M C.C. Braga Parque** - Only 1.74 km away!

**Address:** R. dos Congregados 1, 4710-427 Braga, Portugal  
**Phone:** +351-800780330  
**Opening Hours:** 10:00-23:00 (Every day)

This store is just a short distance from your hotel at Largo Carlos Amarante! It's an easy trip - you could walk there in about 20 minutes or take a quick taxi/Uber ride.

Would you like any other information about this store or assistance with anything else during your shopping trip?


## Google ADK

In [3]:
from dotenv import load_dotenv

ENV_PATH = '/workspace/.env'
load_dotenv(ENV_PATH)

True

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.genai import types

root_agent = LlmAgent(
    model=LiteLlm(model="anthropic/claude-haiku-4-5"),
    name="Jules",
    instruction=SYSTEM_PROMPT,
    tools=[
        McpToolset(
            connection_params=StreamableHTTPConnectionParams(
                url='http://127.0.0.1:8000/mcp',
                timeout=120
            ),
        )
    ],
    generate_content_config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=1024,
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            )
        ]
    )
)
